# Does this transit pass NASA's own tests? Vet a TESS-Keck Survey planet

[Open this notebook in Colab](https://colab.research.google.com/github/Rakiura-Inc/limits-of-habitability/blob/main/notebooks/tess_01_vet_a_candidate.ipynb)

The lab's TESS project checks whether planets that passed TESS's vetting really are planets. This notebook is the first piece of that work: you read NASA's own validation report for one planet, check for nearby stars that could fake its dip, and fill in a vetting sheet the lab can use.

Run the cells in order (Shift+Enter). Boxes headed **Write down** ask you questions. Answer each one in the cell underneath by replacing its `...`, then run that cell: the notebook keeps your answer. The last cell puts all your answers into a single table and saves it. That table is what you hand in on the step.

In [ ]:
# Setup. Run this first. In Colab it installs what the notebook needs.
# If Colab asks you to restart the session afterwards, restart it and run the notebook from the top.
import importlib.util

# habtrain: a small package written for these notebooks. It downloads the data, fits orbits and draws the plots.
if importlib.util.find_spec("habtrain") is None or importlib.util.find_spec("batman") is None:
    %pip install -q "habtrain[tess] @ git+https://github.com/Rakiura-Inc/limits-of-habitability@main"

# rakiura: the optional helper that connects this notebook to your step in Rakiura.
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"

import numpy as np                 # arrays and arithmetic
import matplotlib.pyplot as plt    # plots
import habtrain as ht              # the notebooks' package, used as ht.data, ht.fit, ht.plot and so on

# Which notebook this is, so your hand-in records exactly what you ran.
NOTEBOOK = "notebooks/tess_01_vet_a_candidate.ipynb"
MODE = ht.portal.connect(NOTEBOOK)

> **Connect to Rakiura (optional)**
>
> Connect, and Rakiura ticks off each part as you finish it, with your plots on the step. You still attach your answers table and press Hand in on the step yourself. Skip it and everything still works: you attach the saved files on the step.
>
> To connect, run the cell below, open the link it prints, check the code matches the one shown, and click Allow. In Colab you do this once per session. To skip the click next time, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; your key is in Rakiura under Settings > Notebook key).

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = ht.portal.connect(NOTEBOOK, sign_in=True)

## What can fake a transit

A dip in a star's light is not always a planet. The usual impostors:
- **An eclipsing binary:** two stars that eclipse each other. Their dips often alternate in depth from one to the next (the odd and even dips differ), and there can be a shallower second dip half an orbit later, when the brighter star passes in front (a secondary eclipse).
- **A nearby star:** TESS pixels are 21 arcseconds across, so several stars can share them. If a faint neighbour is an eclipsing binary, its dips blend into the target's light, and the centre of the light shifts during the dip.
- **Scattered light or reflections** inside the camera, which can mimic a dip.

NASA's TESS pipeline (SPOC) runs tests for each of these on every candidate and publishes a **data validation report** for each star. Gaia adds a second check: **RUWE**, a number for how well a single-star model fits a star's motion across the sky. A value well above 1 (commonly, above 1.4) can mean an unseen companion.

## Worked example: TOI-1451 b

In [ ]:
# The star behind TOI-1451 b: its TESS Input Catalog number, position, mass and radius (NASA Exoplanet Archive).
star = ht.data.star("TOI-1451 b")
print(f"TIC {star['tic']}, RA {star['ra']:.4f}, Dec {star['dec']:.4f}")

# NASA's validation reports for this star, one per pipeline run, oldest first.
reports = ht.vet.validation_reports(star["tic"])
for r in reports:
    print(r["file"])
print("\nNewest report (open it in your browser):", reports[-1]["url"] if reports else "none found")

Open the newest report. Its summary page lists, for each signal the pipeline found, the results of its tests. For the vetting sheet, find these four:
- **Odd-even depth:** do the odd and even dips have the same depth?
- **Weak secondary:** is there a second, shallower dip anywhere else in the orbit?
- **Difference image centroid offset:** does the centre of the light move during the dip, and by how much?
- **Ghost diagnostic:** does the dip look like scattered light rather than the star?

The pipeline prints a value for each one. Use these rules to call each a pass or a flag:
- **Odd-even depth:** pass if the odd and even depths differ by less than 3 sigma.
- **Weak secondary:** pass if no secondary dip reaches 3 sigma.
- **Centroid offset:** pass if the light's centre during the dip sits less than 3 sigma from the target star.
- **Ghost diagnostic:** pass if the core statistic is larger than the halo statistic (the dip is strongest on the star itself).

A neighbour matters if it sits within 42 arcseconds (two TESS pixels) and is bright enough to make the whole dip on its own: its G magnitude minus the target's is less than 2.5 × log10(1 ÷ depth), with the depth as a fraction. For a 1,000 parts per million dip that is 7.5 magnitudes.

In [ ]:
# Every Gaia DR3 star within two TESS pixels (42 arcseconds) of the target, closest first, with its RUWE.
neighbours = ht.vet.gaia_neighbours(star["ra"], star["dec"])
print(neighbours.head(8).to_string(index=False))

# The follow-up community's page for this star: imaging and spectra others have taken.
print("\nExoFOP:", ht.vet.exofop_page(star["tic"]))

The first row is the target itself (separation near 0): its RUWE is the number for the vetting sheet. The rows below it are neighbours. A neighbour bright enough and close enough could dilute or fake the dip; G is Gaia's brightness, where a bigger number is fainter.

## Your planet

Choose a planet to vet from the map. You will measure its radius and mass in the next notebook, so pick one you want to spend time on. Pick a different planet from the one you weighed in the bootcamp, so this is new work.

In [ ]:
# The planets you can vet. TOI-669 b is the next notebook's worked example, so it is left out here.
planets = ht.data.planets()
planets = planets[planets.planet != "TOI-669 b"].reset_index(drop=True)
fig = ht.plot.planet_map(planets)
plt.show()

# A menu of the same planets. Choosing one circles it on the map.
chooser = ht.choose.planet(planets, ht.plot.planet_map)

In [ ]:
MY_PICK = ...   # only if the menu did not appear: your planet's number on the map

# Your planet: what you chose in the menu, else the number you typed, else a random one.
planet = ht.choose.resolve(chooser, MY_PICK, planets, "planet", "planet")
ht.record.save("my_planet", planet, "your planet")

# Its star, its validation reports, its Gaia neighbours and its ExoFOP page, as in the worked example.
star = ht.data.star(planet)
reports = ht.vet.validation_reports(star["tic"])
print(f"TIC {star['tic']}: {len(reports)} validation reports. Newest:", reports[-1]["url"] if reports else "none")
neighbours = ht.vet.gaia_neighbours(star["ra"], star["dec"])
print(neighbours.head(8).to_string(index=False))
print("ExoFOP:", ht.vet.exofop_page(star["tic"]))

> **Write down: your vetting sheet**
>
> - Odd-even depth: what does the newest report say, and is it a pass or a flag? Put it in `ODD_EVEN`, for example `"depths agree, pass"`.
> - Weak secondary: what does it say, pass or flag? Put it in `SECONDARY`.
> - Centroid offset: how far does the light move, and pass or flag? Put it in `CENTROID`.
> - Ghost diagnostic: pass or flag? Put it in `GHOST`.
> - Your star's Gaia RUWE (the first row of the neighbours table). Put the number in `RUWE`.
> - The closest neighbour that could matter: its brightness G and distance in arcseconds. Put it in `NEIGHBOUR`.
> - Your call: does this transit pass? And what would you want checked next? Put it in `VERDICT`.
>
> **Where to look:** the newest validation report, the neighbours table, and the ExoFOP page.
>
> Replace each `...` in the cell below with your answer, then run the cell. Leave `...` for anything you skip.

In [ ]:
ODD_EVEN = ...    # what the report says, and pass or flag
SECONDARY = ...
CENTROID = ...
GHOST = ...
RUWE = ...        # a number
NEIGHBOUR = ...   # G and distance of the closest neighbour bright enough to matter, or "none within 42 arcsec"
VERDICT = ...     # your call, and what to check next

# Keep the whole sheet for your hand-in table.
for key, value, what in [("odd_even", ODD_EVEN, "odd-even depth"), ("secondary", SECONDARY, "weak secondary"),
                         ("centroid", CENTROID, "centroid offset"), ("ghost", GHOST, "ghost diagnostic"),
                         ("ruwe", RUWE, "Gaia RUWE"), ("neighbour", NEIGHBOUR, "closest neighbour that could matter"),
                         ("verdict", VERDICT, "your call, and what to check next")]:
    ht.record.save(key, value, what)

**Worth knowing**

These are common working rules, not the lab's own; Michelle may set different ones. Record each value as the report prints it next to your call, so anyone can check your reading quickly.

## Hand in

Run the cell below. It shows your vetting sheet as one table. Check it, then hand in this step in Rakiura, either by pasting the table in or by attaching the saved files. To get the files in Colab, click the folder icon on the left, open `handin`, then `vetting-sheet`, and download them.

In [ ]:
# Your vetting sheet, saved as your hand-in.
ht.record.handin("vetting-sheet", ["my_planet", "odd_even", "secondary", "centroid", "ghost", "ruwe", "neighbour", "verdict"])